In [11]:
import pandas as pd

df = pd.read_csv('../data/raw/customer_orders_raw.csv', parse_dates=['OrderDate'])

In [12]:
snapshot_date = pd.Timestamp('2014-06-30')

rfm = df.groupby('CustomerID').agg(
    FirstName=('FirstName', 'first'),
    LastName=('LastName', 'first'),
    TerritoryName=('TerritoryName', 'first'),
    LastOrderDate=('OrderDate', 'max'),
    Frequency=('SalesOrderID', 'count'),
    Monetary=('TotalDue', 'sum')
).reset_index()

rfm['Recency'] = (snapshot_date - rfm['LastOrderDate']).dt.days

rfm.head()

,CustomerID,FirstName,LastName,TerritoryName,LastOrderDate,Frequency,Monetary,Recency
0,11000,Jon,Yang,Australia,2013-10-03,3,9115.1341,270
1,11001,Eugene,Huang,Australia,2014-05-12,3,7054.1875,49
2,11002,Ruben,Torres,Australia,2013-07-26,3,8966.0143,339
3,11003,Christy,Zhu,Australia,2013-10-10,3,8993.9155,263
4,11004,Elizabeth,Johnson,Australia,2013-10-01,3,9056.5911,272


In [13]:
first_order = df.groupby('CustomerID')['OrderDate'].min().reset_index()
first_order.columns = ['CustomerID', 'FirstOrderDate']

rfm = rfm.merge(first_order, on='CustomerID')
rfm['Tenure'] = (snapshot_date - rfm['FirstOrderDate']).dt.days

rfm[['CustomerID', 'FirstOrderDate', 'Tenure']].head()

,CustomerID,FirstOrderDate,Tenure
0,11000,2011-06-21,1105
1,11001,2011-06-17,1109
2,11002,2011-06-09,1117
3,11003,2011-05-31,1126
4,11004,2011-06-25,1101


In [14]:
rfm['AvgOrderValue'] = rfm['Monetary'] / rfm['Frequency']

rfm[['CustomerID', 'Monetary', 'Frequency', 'AvgOrderValue']].head()

,CustomerID,Monetary,Frequency,AvgOrderValue
0,11000,9115.1341,3,3038.378033
1,11001,7054.1875,3,2351.395833
2,11002,8966.0143,3,2988.671433
3,11003,8993.9155,3,2997.971833
4,11004,9056.5911,3,3018.863700


In [15]:
def avg_purchase_gap(dates):
    dates = dates.sort_values()
    if len(dates) < 2:
        return 0
    gaps = dates.diff().dt.days.dropna()
    return gaps.mean()

purchase_gaps = df.groupby('CustomerID')['OrderDate'].apply(avg_purchase_gap).reset_index()
purchase_gaps.columns = ['CustomerID', 'AvgPurchaseGap']

rfm = rfm.merge(purchase_gaps, on='CustomerID')

rfm[['CustomerID', 'Frequency', 'AvgPurchaseGap']].head(10)

,CustomerID,Frequency,AvgPurchaseGap
0,11000,3,417.5
1,11001,3,530.0
2,11002,3,389.0
3,11003,3,431.5
4,11004,3,414.5
5,11005,3,427.0
6,11006,3,420.5
7,11007,3,400.0
8,11008,3,383.5
9,11009,3,416.5


In [16]:
print(rfm.shape)
rfm.describe()

(19119, 12)


,CustomerID,LastOrderDate,Frequency,Monetary,Recency,FirstOrderDate,Tenure,AvgOrderValue,AvgPurchaseGap
count,19119.000000,19119,19119.000000,19119.000000,19119.000000,19119,19119.000000,19119.000000,19119.000000
mean,20559.000000,2013-12-21 17:34:49.502588928,1.645745,6444.729647,190.267483,2013-07-06 04:15:51.263141376,358.822323,1678.117828,137.026609
min,11000.000000,2011-05-31 00:00:00,1.000000,1.518300,0.000000,2011-05-31 00:00:00,0.000000,1.518300,0.000000
25%,15779.500000,2013-10-10 00:00:00,1.000000,60.752900,85.000000,2013-02-13 00:00:00,145.000000,45.829900,0.000000
50%,20559.000000,2014-01-16 00:00:00,1.000000,606.622900,165.000000,2013-09-25 00:00:00,278.000000,606.622900,0.000000
75%,25338.500000,2014-04-06 00:00:00,2.000000,3119.149400,263.000000,2014-02-05 00:00:00,502.000000,2152.834250,200.000000
max,30118.000000,2014-06-30 00:00:00,28.000000,989184.082000,1126.000000,2014-06-30 00:00:00,1126.000000,151704.902175,1089.000000
std,5519.324234,NaN,1.457054,43756.276004,150.423605,NaN,285.321067,6034.784983,233.030850


In [17]:
rfm['Churned'] = (rfm['Recency'] > 180).astype(int)

rfm['Churned'].value_counts()

Churned
0    10354
1     8765
Name: count, dtype: int64

In [18]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

rfm[['CustomerID', 'Recency', 'R_Score', 'Frequency', 'F_Score', 'Monetary', 'M_Score', 'RFM_Score']].head(10)

,CustomerID,Recency,R_Score,Frequency,F_Score,Monetary,M_Score,RFM_Score
0,11000,270,2,3,5,9115.1341,5,255
1,11001,49,5,3,5,7054.1875,5,555
2,11002,339,1,3,5,8966.0143,5,155
3,11003,263,2,3,5,8993.9155,5,255
4,11004,272,2,3,5,9056.5911,5,255
5,11005,271,2,3,5,8974.0698,5,255
6,11006,259,2,3,5,8971.5283,5,255
7,11007,315,1,3,5,9073.1551,5,155
8,11008,332,1,3,5,8957.4726,5,155
9,11009,264,2,3,5,8940.9197,5,255


In [19]:
def rfm_segment(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal Customers'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New/Promising'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 4:
        return 'At Risk'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)

rfm['Segment'].value_counts()

Segment
Loyal Customers    4655
Needs Attention    3542
Lost               3263
New/Promising      2838
Champions          2454
At Risk            2367
Name: count, dtype: int64

In [20]:
rfm.to_csv('../data/processed/rfm_segmented.csv', index=False)

In [21]:
rfm.head()

,CustomerID,FirstName,LastName,TerritoryName,LastOrderDate,Frequency,Monetary,Recency,FirstOrderDate,Tenure,AvgOrderValue,AvgPurchaseGap,Churned,R_Score,F_Score,M_Score,RFM_Score,Segment
0,11000,Jon,Yang,Australia,2013-10-03,3,9115.1341,270,2011-06-21,1105,3038.378033,417.5,1,2,5,5,255,At Risk
1,11001,Eugene,Huang,Australia,2014-05-12,3,7054.1875,49,2011-06-17,1109,2351.395833,530.0,0,5,5,5,555,Champions
2,11002,Ruben,Torres,Australia,2013-07-26,3,8966.0143,339,2011-06-09,1117,2988.671433,389.0,1,1,5,5,155,At Risk
3,11003,Christy,Zhu,Australia,2013-10-10,3,8993.9155,263,2011-05-31,1126,2997.971833,431.5,1,2,5,5,255,At Risk
4,11004,Elizabeth,Johnson,Australia,2013-10-01,3,9056.5911,272,2011-06-25,1101,3018.863700,414.5,1,2,5,5,255,At Risk
